# Women's local area data
## Debugging 2017 data discrepancy

In testing a new processing approach to allow for greater code reuse through the use of config files, I have identified that data for 2017 in all three of the custodial sentence tables appears to be significantly higher than originally reported.

Initial manual checking against the published Outcomes by Offence data tool suggest that the new processing method is returning accurate data, and that our published tables undercounted the number of custodial sentences for 2017 only.

Through exploration of captures of the Criminal Justice System statistics quarterly: December 2024 webpage by The National Archives web archive, the CSV files first appear on 22 May 2025, after the data tool was published on 15 May. Furthermore, there appear to be filesize differences between the raw dataset which is currently available and downloaded through this data pipeline, and the CSV which has been [manually downloaded from the web archive](https://webarchive.nationalarchives.gov.uk/ukgwa/20250522181645/https://www.gov.uk/government/statistics/criminal-justice-system-statistics-quarterly-december-2024) (dated 22 May 2025).

The aim of this notebook is to explore whether there are differences in the two CSV files, and whether this could explain the discrepancy in the 2017 data that we originally reported.

## Imports

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('data/raw/2025/sentence_outcomes_2017_2024.csv')
df.head()

,Year,Quarter,Month,Person/Other,Sex,Age Group,Age Range,Ethnicity,Detailed Ethnicity,Police Force Area,...,Period of Driving Disqualification,Motoring Offence Flag,Sentenced,custody_rate_filter,acsl_filter,acsl_months,avg_fine_filter_persons,avg_fine_filter_companies,fine_amount_persons,fine_amount_companies
0,2017,Q1 Jan-Mar,01: January,01: Person,02: Male,03: Adults,09: 40 to 49 (2017 onwards),04: White,13: White British,Thames Valley,...,Not applicable,Non-motoring Offence,1,0,0,0,1,0,120,0
1,2017,Q1 Jan-Mar,03: March,01: Person,02: Male,03: Adults,07: 25 to 29 (2017 onwards),04: White,16: Any other white background,Cambridgeshire,...,Not applicable,Non-motoring Offence,1,0,0,0,1,0,600,0
2,2017,Q1 Jan-Mar,03: March,01: Person,02: Male,03: Adults,09: 40 to 49 (2017 onwards),04: White,13: White British,Humberside,...,Not applicable,Non-motoring Offence,1,1,1,5,0,0,0,0
3,2017,Q1 Jan-Mar,02: February,01: Person,02: Male,02: Young adults,04: 18 to 20,04: White,13: White British,Northumbria,...,02: 6 months,Motoring Offence,1,0,0,0,0,0,0,0
4,2017,Q1 Jan-Mar,02: February,01: Person,02: Male,03: Adults,11: 60 to 69 (2017 onwards),06: Unknown,19: Unknown,Leicestershire,...,Not applicable,Motoring Offence,2,0,0,0,2,0,80,0


In [3]:
df_old = pd.read_csv('data/raw/2025/sentence_outcomes_2017_2024_old.csv')
df_old.head()

,Year,Quarter,Month,Person/Other,Sex,Age Group,Age Range,Ethnicity,Detailed Ethnicity,Police Force Area,...,Period of Driving Disqualification,Motoring Offence Flag,Sentenced,custody_rate_filter,acsl_filter,acsl_months,avg_fine_filter_persons,avg_fine_filter_companies,fine_amount_persons,fine_amount_companies
0,2017,Q1 Jan-Mar,03: March,01: Person,02: Male,03: Adults,08: 30 to 39 (2017 onwards),06: Unknown,19: Unknown,West Mercia,...,Not applicable,Motoring Offence,1,0,0,0,1,0,220,0
1,2017,Q1 Jan-Mar,01: January,01: Person,02: Male,03: Adults,05: 21 to 24,04: White,13: White British,Northumbria,...,Not applicable,Non-motoring Offence,3,0,0,0,0,0,0,0
2,2017,Q1 Jan-Mar,01: January,01: Person,02: Male,03: Adults,10: 50 to 59 (2017 onwards),04: White,16: Any other white background,Thames Valley,...,Not applicable,Motoring Offence,1,0,0,0,1,0,160,0
3,2017,Q1 Jan-Mar,03: March,01: Person,02: Male,03: Adults,08: 30 to 39 (2017 onwards),04: White,13: White British,Derbyshire,...,Not applicable,Non-motoring Offence,1,0,0,0,1,0,400,0
4,2017,Q1 Jan-Mar,01: January,01: Person,01: Female,03: Adults,08: 30 to 39 (2017 onwards),06: Unknown,19: Unknown,Surrey,...,Not applicable,Motoring Offence,1,0,0,0,1,0,53,0


## Checking dataframe shapes

In [5]:
df.shape

(4317508, 29)

In [8]:
df_old.shape

(4266613, 29)

In [9]:
df.shape[0] - df_old.shape[0]

50895

There is a clear discrepancy in shapes here with a difference of nearly 51,000 rows of data.

I am most interested in whether any of those missing rows are for data in 2017, given that is the year where I have the discrepancy.

In [11]:
df.query('Year == 2017').shape

(602884, 29)

In [13]:
df_old.query('Year == 2017').shape

(551989, 29)

In [10]:
df.query('Year == 2017').shape[0] - df_old.query('Year == 2017').shape[0]

50895

This seems to suggest that **ALL** the missing rows are for 2017! Let me check whether there are any issues for the other years.

In [14]:
for year in df['Year'].unique():
    print(f"Year: {year}")
    print(f"New data shape: {df.query('Year == @year').shape}")
    print(f"Old data shape: {df_old.query('Year == @year').shape}")
    print(f"Difference: {df.query('Year == @year').shape[0] - df_old.query('Year == @year').shape[0]}")
    print()

Year: 2017
New data shape: (602884, 29)
Old data shape: (551989, 29)
Difference: 50895

Year: 2018
New data shape: (578874, 29)
Old data shape: (578874, 29)
Difference: 0

Year: 2019
New data shape: (570958, 29)
Old data shape: (570958, 29)
Difference: 0

Year: 2020
New data shape: (442432, 29)
Old data shape: (442432, 29)
Difference: 0

Year: 2021
New data shape: (518268, 29)
Old data shape: (518268, 29)
Difference: 0

Year: 2022
New data shape: (524408, 29)
Old data shape: (524408, 29)
Difference: 0

Year: 2023
New data shape: (532924, 29)
Old data shape: (532924, 29)
Difference: 0

Year: 2024
New data shape: (546760, 29)
Old data shape: (546760, 29)
Difference: 0



That appears to be pretty definitive! Let's see when the dataset was updated as I had completed my data analysis by 22 July 2025.

Manual checking via the web archive suggests that a new zip file, containing the CSVs was uploaded on 4 August 2025, called `Outcomes_by_offence_2017_2024`, this is different to the original `obo_2017_2024`. I've moved the file into `data/raw/2025` to check if this aligns with the data I've processed.

In [15]:
df_august = pd.read_csv('data/raw/2025/sentence_outcomes_2017_2024_august.csv')
df_august.shape

(4317508, 29)

In [16]:
for year in df['Year'].unique():
    print(f"Year: {year}")
    print(f"New data shape: {df.query('Year == @year').shape}")
    print(f"Old data shape: {df_august.query('Year == @year').shape}")
    print(f"Difference: {df.query('Year == @year').shape[0] - df_august.query('Year == @year').shape[0]}")
    print()

Year: 2017
New data shape: (602884, 29)
Old data shape: (602884, 29)
Difference: 0

Year: 2018
New data shape: (578874, 29)
Old data shape: (578874, 29)
Difference: 0

Year: 2019
New data shape: (570958, 29)
Old data shape: (570958, 29)
Difference: 0

Year: 2020
New data shape: (442432, 29)
Old data shape: (442432, 29)
Difference: 0

Year: 2021
New data shape: (518268, 29)
Old data shape: (518268, 29)
Difference: 0

Year: 2022
New data shape: (524408, 29)
Old data shape: (524408, 29)
Difference: 0

Year: 2023
New data shape: (532924, 29)
Old data shape: (532924, 29)
Difference: 0

Year: 2024
New data shape: (546760, 29)
Old data shape: (546760, 29)
Difference: 0



Excellent — there doesn't appear to be any difference in the shape, so I'm pretty sure this is the source of the discrepancy. Let's try and run the additional processing steps for the old dataset and see if I can replicate our published output.

## Reproducing processing steps using old dataset

Just using the processing steps that I need for checking.

In [20]:
import logging
from src.configuration import load_config
from src.data.processing import (filter_sentence_type,
                                 group_pfa_sentence_outcome,
                                 filter_sentence_length,
                                 make_custody_tables)

In [ ]:
def process_custodial_sentences(config: dict) -> None:

    logging.info(
        "Starting processing for %s analysis",
        config["analysis"]["id"],
    )

    filter_sentence_type.main(config=config)
    group_pfa_sentence_outcome.main(config=config)
    filter_sentence_length.main(config=config)
    make_custody_tables.main(config=config)

    logging.info("Data processing pipeline completed successfully.")

Temporarily updating my 2025 config file to use `data/raw/2025/sentence_outcomes_2017_2024_old.csv` rather than the current version, and the data paths for testing.

In [22]:
config = load_config(2025)
config['data']['filenames']['outcomes_by_offence'] = "sentence_outcomes_2017_2024_old.csv"
config['data']['filenames']['outcomes_by_offence']

'sentence_outcomes_2017_2024_old.csv'

In [25]:
config['paths']['interim'] = "data/interim/2025/testing"
config['paths']['processed'] = "data/processed/2025/testing"
config['paths']

{'raw': 'data/raw/2025',
 'interim': 'data/interim/2025/testing',
 'processed': 'data/processed/2025/testing',
 'figures': 'reports/2025/figures',
 'tests': 'data/tests/2025'}

In [26]:
process_custodial_sentences(config=config)

2026-10-15 15:01:50,469 - INFO - Starting processing for 2025 analysis
2026-10-15 15:01:50,528 - INFO - Loading outcomes by offence data...
2026-10-15 15:03:20,221 - INFO - Loaded data from data/raw/2025/sentence_outcomes_2017_2024_old.csv
2026-10-15 15:04:13,520 - INFO - Loaded data from data/raw/2025/sentence_outcomes_2010_2016.csv
2026-10-15 15:04:20,609 - INFO - Processing data...
2026-10-15 15:04:43,169 - INFO - Applying filters...
2026-10-15 15:04:43,178 - INFO - Include filter on column 'sex' with values: ['Female']
2026-10-15 15:04:45,104 - INFO - Include filter on column 'outcome' with values: ['Immediate Custody', 'Community Sentence', 'Suspended Sentence']
2026-10-15 15:04:45,340 - INFO - Include filter on column 'age_group' with values: ['Adults', 'Young adults']
2026-10-15 15:04:45,477 - INFO - Exclude filter on column 'pfa' with values: ['Not known']
2026-10-15 15:04:45,633 - INFO - Data filtered.
2026-10-15 15:04:46,561 - INFO - Data loaded and processed successfully.
20

## Checking totals against those we published

In [34]:
custody_all = pd.read_csv('data/processed/2025/testing/PFA_custodial_sentences_all_FINAL.csv')
custody_6months = pd.read_csv('data/processed/2025/testing/PFA_custodial_sentences_6_months_FINAL.csv')
custody_12months = pd.read_csv('data/processed/2025/testing/PFA_custodial_sentences_12_months_FINAL.csv')

In [35]:
dataframes = {
    "custody_all": custody_all,
    "custody_6months": custody_6months,
    "custody_12months": custody_12months,
}

for name, df in dataframes.items():
    df.drop(columns=['per_change_2014'], inplace=True)
    print(f"\n{name}")
    print(df.sum(numeric_only=True))


custody_all
2014    7394
2015    7238
2016    7262
2017    5398
2018    6580
2019    5862
2020    4354
2021    4154
2022    4026
2023    4830
2024    5738
dtype: int64

custody_6months
2014    5001
2015    4881
2016    4937
2017    4436
2018    4389
2019    3808
2020    2726
2021    2369
2022    2424
2023    3072
2024    3790
dtype: int64

custody_12months
2014    5676
2015    5536
2016    5582
2017    4726
2018    4974
2019    4374
2020    3254
2021    2841
2022    2829
2023    3483
2024    4301
dtype: int64


These all appear to match up perfectly with our previously published stats, so that appears to have been the source of the discrepancy.